# PandaPickCube — обучение в Google Colab

Ноутбук самодостаточный: не клонирует репозиторий и не вызывает `train.py` — логика взята из `config.py` и `train.py` и встроена ниже.

Ноутбук использует PyTorch из образа Colab и фиксированные версии MuJoCo, Playground, RSL-RL и JAX с CUDA 12.

## Текущий эксперимент: `max_iters_500`

- `EXP_NAME = "max_iters_500"`
- `NUM_ENVS = 4096` — оставлено без изменений
- `MAX_ITERS = 1001` — уменьшено с 1001
- `SAVE_INTERVAL = 50` — оставлено без изменений

Относительно исходной конфигурации эксперимента изменён только `MAX_ITERS`; имя эксперимента задано отдельно, чтобы результаты записывались в новую папку.

## Перед запуском

1. **Runtime → Change runtime type → GPU**.
2. Выполняйте ячейки сверху вниз. Установка занимает несколько минут.
3. Если до установки вы уже импортировали JAX или PyTorch, перезапустите сеанс через **Runtime → Restart session**. Затем продолжайте с настройки рабочей папки.

Чекпоинты и логи сохраняются в `logs/` относительно текущей рабочей директории. По умолчанию это `/content`; после отключения/сброса среды Colab файлы в `/content` могут исчезнуть. Для постоянного хранения подключите Google Drive в следующей ячейке и задайте `LOG_ROOT` в ячейке 3.

## 1. Установка зависимостей

Установка сохраняет PyTorch из образа Colab и проверяет, что JAX видит GPU. Предустановленный плагин JAX для CUDA 13 удаляется из текущей сессии, чтобы избежать конфликта с CUDA 12.

In [1]:
from importlib.metadata import version
import subprocess
import sys

subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
    check=True,
)

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'jax-cuda13-plugin', 'jax-cuda13-pjrt'],
    check=True,
)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--quiet',
        f'torch=={version("torch")}', 'playground==0.1.0', 'rsl-rl-lib==5.5.2',
        'jax[cuda12]==0.6.2', 'mujoco==3.5.0', 'mujoco-mjx==3.5.0',
        'numpy==2.2.6', 'tensorboard==2.21.0',
    ],
    check=True,
)

print(
    subprocess.check_output(
        [sys.executable, '-c', 'import jax; print(jax.__version__, jax.devices("gpu"))'],
        text=True,
    )
)


0.6.2 [CudaDevice(id=0)]



## 2. (Опционально) Google Drive

Раскомментируйте, смонтируйте диск, затем в следующей ячейке задайте `LOG_ROOT` на путь вроде `/content/drive/MyDrive/colab_panda`.

In [2]:
# from google.colab import drive
# drive.mount("/content/drive")

## 3. Рабочая папка и переменные окружения

`MUJOCO_GL=egl` — headless-рендер на Colab. Логи пишутся в `logs/` внутри `LOG_ROOT`.

In [3]:
import os

LOG_ROOT = "/content"
# Например: "/content/drive/MyDrive/colab_panda" после монтирования Google Drive
os.makedirs(LOG_ROOT, exist_ok=True)
os.chdir(LOG_ROOT)

os.environ["MUJOCO_GL"] = "egl"
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

print("cwd:", os.getcwd())


cwd: /content


## 4. Обучение

Ниже встроены `config.py` и логика из `train.py`. Для быстрой проверки можно временно задать `EXP_NAME = "colab_smoke"`, `NUM_ENVS = 64`, `MAX_ITERS = 2` и `SAVE_INTERVAL = 1`. На T4 при нехватке памяти в полном прогоне уменьшите `NUM_ENVS` до `1024` или `512`.

**Для текущего эксперимента не меняйте эти параметры:** `EXP_NAME = "max_iters_500"`, `NUM_ENVS = 4096`, `MAX_ITERS = 500`, `SAVE_INTERVAL = 50`.

In [4]:
"""Встроенные config.py + train.py для Colab (без внешнего репозитория)."""

import json
import os
import time
from datetime import datetime
from types import SimpleNamespace

import jax
import torch
from mujoco_playground import registry, wrapper_torch
from mujoco_playground.config import manipulation_params
from rsl_rl.runners import OnPolicyRunner

jax.config.update("jax_default_matmul_precision", "highest")

# --- Параметры запуска (аналог аргументов CLI из train.py) ---
NUM_ENVS = 4096
MAX_ITERS = 1001
SAVE_INTERVAL = 50
SEED = 1

EXP_NAME = "qpos_strong"
RESUME = None
DEVICE = "cuda:0"

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch не видит GPU. Проверьте тип среды выполнения Colab.")
print("PyTorch GPU:", torch.cuda.get_device_name(0))
print("JAX GPU:", jax.devices("gpu"))

# --- config.py ---
ENV_NAME = "PandaPickCube"

ACTOR_HIDDEN_DIMS = [512, 256, 128]
CRITIC_HIDDEN_DIMS = [512, 256, 128]
ACTIVATION = "elu"
INIT_NOISE_STD = 1.0

def build_runner_cfg(obs_size) -> dict:
    cfg = manipulation_params.rsl_rl_config(ENV_NAME).to_dict()
    _policy = cfg.pop("policy", {})
    cfg["actor"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": ACTOR_HIDDEN_DIMS,
        "activation": ACTIVATION,
        "distribution_cfg": {
            "class_name": "rsl_rl.modules.distribution.GaussianDistribution",
            "init_std": INIT_NOISE_STD,
        },
    }
    cfg["critic"] = {
        "class_name": "rsl_rl.models.mlp_model.MLPModel",
        "hidden_dims": CRITIC_HIDDEN_DIMS,
        "activation": ACTIVATION,
    }
    if isinstance(obs_size, dict):
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["privileged_state"]}
    else:
        cfg["obs_groups"] = {"actor": ["state"], "critic": ["state"]}
    return cfg

def resolve_log_dir(exp_name, resume) -> str:
    if exp_name:
        name = exp_name
    elif resume:
        name = os.path.basename(os.path.dirname(os.path.abspath(resume)))
    else:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = f"{ENV_NAME}_{timestamp}"
    log_dir = os.path.join("logs", name)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir

args = SimpleNamespace(
    exp_name=EXP_NAME,
    num_envs=NUM_ENVS,
    max_iters=MAX_ITERS,
    device=DEVICE,
    resume=RESUME,
    save_interval=SAVE_INTERVAL,
    seed=SEED,
)

device = args.device
device_rank = int(device.split(":")[-1]) if "cuda" in device else 0

log_dir = resolve_log_dir(args.exp_name, args.resume)

print(f"Среда:            {ENV_NAME}")
print(f"Устройство:       {device}")
print(f"Параллельных сред: {args.num_envs}")
print(f"Итерации:         {args.max_iters}")
print(f"Лог/чекпоинты:    {log_dir}")
print()

env_cfg = registry.get_default_config(ENV_NAME)

# Эксперимент B: усиливаем награду за сохранение стартовой позы
env_cfg.reward_config.scales.robot_target_qpos = 30.0

raw_env = registry.load(ENV_NAME, config=env_cfg)

brax_env = wrapper_torch.RSLRLBraxWrapper(
    raw_env,
    num_actors=args.num_envs,
    seed=args.seed,
    episode_length=env_cfg.episode_length,
    action_repeat=1,
    device_rank=device_rank,
)
brax_env.cfg = {}

cfg_dict = build_runner_cfg(raw_env.observation_size)
cfg_dict["seed"] = args.seed
cfg_dict["max_iterations"] = args.max_iters
cfg_dict["save_interval"] = args.save_interval

config_path = os.path.join(log_dir, "config.json")
with open(config_path, "w") as f:
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Конфиг сохранён: {config_path}")

runner = OnPolicyRunner(brax_env, cfg_dict, log_dir, device=device)

if args.resume:
    print(f"Загрузка чекпоинта: {args.resume}")
    runner.load(args.resume)

print("=" * 60)
print("  Начинаю обучение...")
print("=" * 60)
print()

t_start = time.time()
runner.learn(
    num_learning_iterations=args.max_iters,
    init_at_random_ep_len=False,
)
elapsed = time.time() - t_start

print()
print("=" * 60)
print(f"  Обучение завершено за {elapsed:.0f} с ({elapsed / 60:.1f} мин)")
print(f"  Логи и чекпоинты: {log_dir}")
print("=" * 60)


PyTorch GPU: Tesla T4
JAX GPU: [CudaDevice(id=0)]
Среда:            PandaPickCube
Устройство:       cuda:0
Параллельных сред: 4096
Итерации:         1001
Лог/чекпоинты:    logs/box_target_strong

mujoco_menagerie not found. Downloading...


Cloning mujoco_menagerie: ██████████| 100/100 [01:00<00:00]


Checking out commit 1b86ece576591213e2b666ebf59508454200ca97
Successfully downloaded mujoco_menagerie
Device -- cuda:0
Key device -- {CudaDevice(id=0)}
obs_shape: 66
JITing reset and step
Done JITing reset and step
Конфиг сохранён: logs/box_target_strong/config.json
--------------------------------------------------------------------------------
Resolved observation sets: 
	 actor :  ['state']
	 critic :  ['state']
--------------------------------------------------------------------------------
Actor Model: MLPModel(
  (obs_normalizer): Identity()
  (distribution): GaussianDistribution()
  (mlp): MLP(
    (0): Linear(in_features=66, out_features=512, bias=True)
    (1): ELU(alpha=1.0)
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ELU(alpha=1.0)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ELU(alpha=1.0)
    (6): Linear(in_features=128, out_features=8, bias=True)
  )
)
Critic Model: MLPModel(
  (obs_normalizer): Identity()
  (mlp): MLP(

/usr/local/lib/python3.13/dist-packages/jax/_src/interpreters/xla.py:119: RuntimeWarning: overflow encountered in cast
  return np.asarray(x, dtypes.canonicalize_dtype(x.dtype))


Выходные данные были обрезаны до нескольких последних строк (5000).
                         Iteration time: 4.34s
                           Time elapsed: 0:57:30
                                    ETA: 0:15:54

################################################################################
                          Learning iteration 784/1001                            

                            Total steps: 128614400 
                       Steps per second: 38377 
                        Collection time: 3.659s 
                          Learning time: 0.610s 
                        Mean value loss: 194.3923
                    Mean surrogate loss: -0.0097
                      Mean entropy loss: 32.8388
                            Mean reward: 803.68
                    Mean episode length: 150.00
                        Mean action std: 14.76
                Mean episode box_target: 0.0504
               Mean episode gripper_box: 0.7175
        Mean episode no_floor_collisi

## 5. (Опционально) TensorBoard

Логи: `logs/<имя_эксперимента>/` относительно `LOG_ROOT`.

In [5]:
# %load_ext tensorboard
# %tensorboard --logdir logs